In [ ]:
# CELL 1: Load schemes and extract original FAQs

import json
import pandas as pd
from pathlib import Path

INPUT_FILE = Path("data/golden_dataset/selected_schemes.json")
OUTPUT_DIR = Path("data/golden_dataset/final")
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

with open(INPUT_FILE, "r", encoding="utf-8") as f:
    schemes = json.load(f)

records = []

for scheme in schemes:
    scheme_id = scheme["scheme_id"]

    for i, faq in enumerate(scheme.get("faqs", []), start=1):
        if faq.get("question") and faq.get("answer"):
            records.append({
                "test_id": f"GD_{len(records)+1:06d}",
                "faq_id": f"{scheme_id}_FAQ_{i:03d}",
                "scheme_id": scheme_id,
                "scheme_name": scheme.get("name"),
                "question": faq["question"],
                "ground_truth_answer": faq["answer"],
                "categories": scheme.get("categories", []),
                "level": scheme.get("level"),
                "source_url": faq.get("url") or scheme.get("myscheme_url"),
                "relevant_scheme_ids": [scheme_id],
                "relevant_doc_ids": [],
                "answerable": True
            })

golden_df = pd.DataFrame(records)

print("Total Golden Dataset records:", len(golden_df))

In [ ]:
# CELL 2: Validate Golden Dataset

print("Missing values:\n", golden_df.isnull().sum())

print("Duplicate test IDs:", golden_df["test_id"].duplicated().sum())
print("Duplicate FAQ IDs:", golden_df["faq_id"].duplicated().sum())

print("Duplicate questions:", golden_df.duplicated(
    subset=["scheme_id", "question"]
).sum())

print("Total unique schemes:", golden_df["scheme_id"].nunique())

In [ ]:
# CELL 3: Save Golden Dataset

golden_df.to_json(
    OUTPUT_DIR / "golden_dataset.json",
    orient="records",
    indent=4,
    force_ascii=False
)

golden_df.to_csv(
    OUTPUT_DIR / "golden_dataset.csv",
    index=False,
    encoding="utf-8-sig"
)

print("Golden Dataset saved successfully!")

In [ ]:
# CELL 1: Randomly split Golden Dataset into train, validation and test

import pandas as pd
from sklearn.model_selection import train_test_split

df = pd.read_json("G:\\SEVA_Ai\\SEVA_Ai\\notebooks\\golden_dataset\\final\\golden_dataset.json")

train_df, temp_df = train_test_split(
    df, test_size=0.30, random_state=42
)

val_df, test_df = train_test_split(
    temp_df, test_size=0.50, random_state=42
)

OUTPUT = "G:\\SEVA_Ai\SEVA_Ai\\notebooks\\golden_dataset\\final\\Final_dataset_splits"

for name, data in [
    ("train", train_df),
    ("validation", val_df),
    ("test", test_df)
]:
    data.to_json(
        f"{OUTPUT}/{name}.json",
        orient="records",
        indent=4,
        force_ascii=False
    )

print(f"Train: {len(train_df)}")
print(f"Validation: {len(val_df)}")
print(f"Test: {len(test_df)}")